# 问题1：第一阶段 A——真实来源小规模采集与核验

**适用内核：Python for Data Analysis (pda312)，Python 3.12。**

依赖：Python 标准库、requests；表格显示另用你已安装的 pandas。不需要安装 UMAP 或 R 包。

按单元格顺序运行。只对 2026-11-10 做 8 组方向明确的列车查询，尚不生成完整铁路时间矩阵。
输入：项目 `config/research_protocol.json` 和官方公开查询响应。
输出：每次独立的 `data/raw/user_pilot_*` 与 `data/processed/user_pilot_*`。
真实数据可能随查询时间变化；不得按助手测试计数补写结果。

本站点查询有访问限制，代码每次请求至少间隔 2 秒；遇到拒绝、异常响应或网络错误立即停止，无自动重试。
先保存本 Notebook，运行后再次保存。不得用助手访问记录充当本 Notebook 的运行输出。


In [1]:
# 问题1—辅助代码0：确认本次内核、路径与协议
import sys, json, csv, importlib
from pathlib import Path
import pandas as pd

ROOT = Path(r"D:\yangz\大三上\机器学习\第二次作业\railway_analysis")
EXPECTED = Path(r"D:\anaconda\envs\pda312\python.exe")
assert Path(sys.executable).resolve() == EXPECTED.resolve(), "请切换到 pda312 内核"
sys.path.insert(0, str(ROOT / "scripts"))
import stage1_collect
stage1_collect = importlib.reload(stage1_collect)
protocol = json.loads((ROOT / "config/research_protocol.json").read_text(encoding="utf-8"))
print("解释器：", sys.executable)
print("研究日期：", protocol["query_date"], "；种子：", protocol["seed"])
print("本批没有随机算法；这里只记录后续实验统一种子。")


解释器： D:\anaconda\envs\pda312\python.exe
研究日期： 2026-11-10 ；种子： 20261110
本批没有随机算法；这里只记录后续实验统一种子。


## 问题1—核心代码1：采集并保留原始证据

功能：读取官方车站字典、核对官方字段映射，执行 8 组查询，逐条检查列车记录。
输入：冻结协议及实际 HTTP 响应。完整实现位于 `scripts/stage1_collect.py`。
应截图：下方代码、每组查询状态、最终计数。`validated_records` 表示字段检查通过，不表示全国网络覆盖完成。
保存文件：原始 `.bin`、`source_manifest.json`、配置及代码快照、`run_metadata.json`、列车 CSV 和质量 CSV。
若发生错误，不要循环重试；继续查看质量表，并回传原始错误。


In [2]:
# 问题1—核心代码1：真实请求→保存原文→字段验证→质量记录
# actor="user" 表示此单元格由你在本地实际运行。
summary = stage1_collect.collect_pilot(actor="user")
OUT = Path(summary["output_dir"])
for name in ["run_id", "query_date", "node_count", "attempted_queries",
             "validated_queries", "train_record_count", "issue_count", "fatal_error"]:
    print(f"{name}: {summary[name]}")
print("输出目录：", OUT)
print("matrix_ready:", summary["matrix_ready"], "（本批不是完整矩阵）")


P01 北京南 → 上海虹桥: validated_records, valid=53, rejected=0
P02 上海虹桥 → 北京南: validated_records, valid=54, rejected=0
P03 广州南 → 香港西九龙: validated_records, valid=63, rejected=0
P04 香港西九龙 → 广州南: validated_records, valid=61, rejected=0
P05 北京西 → 西宁: validated_records, valid=7, rejected=0
P06 西宁 → 拉萨: validated_records, valid=6, rejected=0
P07 广州 → 海口: validated_records, valid=5, rejected=0
P08 广州南 → 横琴: no_direct_train_returned_not_unreachable, valid=0, rejected=0
run_id: user_pilot_20261105_220920_089173
query_date: 2026-11-10
node_count: 34
attempted_queries: 8
validated_queries: 7
train_record_count: 249
issue_count: 0
fatal_error: None
输出目录： D:\yangz\大三上\机器学习\第二次作业\railway_analysis\data\processed\user_pilot_20261105_220920_089173
matrix_ready: False （本批不是完整矩阵）


## 问题1—核心代码2：核对每组查询和清洗记录

功能：区分有效记录、空返回、请求错误和未执行，检查隔离/重复记录。
输入：本次采集生成的质量 CSV。应截图：代码和 8 行质量表，有清洗问题时加截问题表。
保存文件：这些表已由核心代码1保存，无需手工抄录。
空返回只能说明该次查询未返回直达列车，不能认定铁路不可达。


In [3]:
# 问题1—核心代码2：查询覆盖与清洗检查
quality = pd.read_csv(OUT / "pilot_query_quality.csv", encoding="utf-8-sig")
issues = pd.read_csv(OUT / "cleaning_issues.csv", encoding="utf-8-sig")
display(quality[["query_id", "from_station", "to_station", "status",
                 "raw_count", "valid_count", "rejected_count", "duplicate_count"]])
print("清洗问题条数：", len(issues))
if not issues.empty:
    display(issues)
if summary["fatal_error"]:
    print("需回传的错误：", summary["fatal_error"])


,query_id,from_station,to_station,status,raw_count,valid_count,rejected_count,duplicate_count
0,P01,北京南,上海虹桥,validated_records,53,53,0,0
1,P02,上海虹桥,北京南,validated_records,54,54,0,0
2,P03,广州南,香港西九龙,validated_records,63,63,0,0
3,P04,香港西九龙,广州南,validated_records,61,61,0,0
4,P05,北京西,西宁,validated_records,7,7,0,0
5,P06,西宁,拉萨,validated_records,6,6,0,0
6,P07,广州,海口,validated_records,5,5,0,0
7,P08,广州南,横琴,no_direct_train_returned_not_unreachable,0,0,0,0


清洗问题条数： 0


## 问题1—核心代码3：用真实记录复核跨日历时和实际车站

功能：从本次列车表独立复核“到达时刻−出发时刻＝官方历时分钟数”；统计同城扩展返回和始发日期差异。
输入：本次 `pilot_train_records.csv`。应截图：代码及计数/样例输出；如表太宽，可分截图，CSV 原件同时保留。
注意：列车始发日期与旅客上车日期可能不同，代码不把两者混用。下面的展示不是最短行程分析。


In [4]:
# 问题1—核心代码3：复核分钟单位、跨日日期与实际端点
trains = pd.read_csv(OUT / "pilot_train_records.csv", encoding="utf-8-sig")
if trains.empty:
    print("没有可核验的列车记录，请回传质量表和错误。")
else:
    elapsed = (pd.to_datetime(trains["arrival_at"]) -
               pd.to_datetime(trains["departure_at"])).dt.total_seconds() / 60
    assert (elapsed == trains["duration_min"]).all(), "跨日历时复核未通过"
    same_station = trains["same_as_requested_stations"].astype(str).str.lower().eq("true")
    earlier_origin = trains["origin_service_date"] < trains["query_date"]
    print("历时复核通过条数：", len(trains))
    print("同城其他站返回条数：", int((~same_station).sum()))
    print("列车始发日期早于上车日期条数：", int(earlier_origin.sum()))
    sample = trains.loc[earlier_origin] if earlier_origin.any() else trains.head(4)
    display(sample[["query_id", "train_code", "from_name", "to_name",
                    "origin_service_date", "departure_at", "arrival_at", "duration_min"]].head(4))


历时复核通过条数： 249
同城其他站返回条数： 86
列车始发日期早于上车日期条数： 9


,query_id,train_code,from_name,to_name,origin_service_date,departure_at,arrival_at,duration_min
107,P03,G897,广州南,香港西九龙,2026-11-09,2026-11-10T06:27:00+08:00,2026-11-10T07:19:00+08:00,52
238,P06,Z323,西宁,拉萨,2026-11-09,2026-11-10T11:42:00+08:00,2026-11-11T07:24:00+08:00,1182
240,P06,Z21,西宁,拉萨,2026-11-09,2026-11-10T15:27:00+08:00,2026-11-11T11:46:00+08:00,1219
241,P06,Z265,西宁,拉萨,2026-11-09,2026-11-10T18:19:00+08:00,2026-11-11T14:53:00+08:00,1234


## 回传材料与阶段边界

请保存执行后的 Notebook，回传核心代码1—3及其实际输出截图，以及本次目录下的：

- `pilot_summary.json`
- `pilot_query_quality.csv`
- `pilot_train_records.csv`
- `cleaning_issues.csv`

保留完整 `data/raw/user_pilot_*` 原始目录。请用自己的话说明：为何“无直达”不等于“不可达”；为何跨日列车不能直接用到达钟点减出发钟点。理解说明用于之后的 AI 工具使用记录。

本批尚未补齐城市坐标、完整中转网络、澳门接驳及台湾连接的正式来源，也未生成 34×34 矩阵。核验完成后继续第一阶段 B；不自动进入降维或报告制作。
